 ## Saving you derived datasets
 Create a a file similar to the sturcture of the aggregated data <br>
please use the pescribed names for the data set and variable name<br>
Best is to mount the nci folder so you can write the data directly on the cluster.<br>
to have acces to the data log into nci and then create symbolic links to the follwing folders: e.g.<br>
`ln -s /g/data/au88/ismip6_hackathon . ` <br>
and `ln -s /g/data/au88/ismip6/2300/forcings/ ismip6_2300`
 <br>
your mnt command can now be: <br>
`"sshfs -o follow_symlinks <username>@gadi.nci.org.au:/home/565/<username> mnt/ncihome"`

In [1]:
import numpy as np
import xarray as xr
import os
computed_name = 'ComputedScalars'
mnt_pth = '/Users/jbec0008/mnt/ncihome/' #mount path
pth_calc_output = mnt_pth + 'ismip6_hackathon/' #write folder
pth_ismip6 =mnt_pth + 'ismip6_2300/' #read folder

dir_data_aggr ='calc_output_datasets' # read folder to aggegrated data of Seroussi et al.
expnames = ['expAE02', 'expAE03', 'expAE04', 'expAE05']




### E.g. loading data from cluster onto you local notebook

In [2]:
### for insance load imbie files to have a mask for the regions:
d_region = xr.open_dataset(pth_ismip6+'masks/sectors_8km.nc')

In [3]:
print('basin numbers are:', np.unique(d_region.sectors))

basin numbers are: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18.]




### 0. get all model names

In [5]:
# get all experiments and model runs ( not just the main experiment)
dic_modelnames = {}
for iexp, expname in enumerate(expnames):
    model_names_endfile =[]

    file_exp = os.listdir(f"{computed_name}/{expname}/shelfmelt")
    file_exp = [file for file in file_exp if len(file) >= 3]

    for ifile in reversed(range(len(file_exp))):
        if len(file_exp[ifile]) < 3:
            del file_exp[ifile]
    for ifile in file_exp:
        model_names_endfile.append(ifile.split('AIS_')[-1])
    dic_modelnames[iexp]=model_names_endfile
    
model_names_endfile

['VUW_PISM1_s3_expAE05.nc',
 'IMAU_UFEMISM3_expAE05.nc',
 'NCAR_CISM1_expAE05.nc',
 'VUW_PISM2_expAE05.nc',
 'LSCE_GRISLI2_expAE05.nc',
 'NORCE_CISM4-MAR364-ERA-t1-nonlocal_expAE05.nc',
 'NORCE_CISM3-MAR364-ERA-t1_expAE05.nc',
 'VUW_PISM2_s3_expAE05.nc',
 'NORCE_CISM4-MAR364-ERA-t1-local_expAE05.nc',
 'IMAU_UFEMISM1_expAE05.nc',
 'VUW_PISM2_s4_expAE05.nc',
 'ULB_fETISh-KoriBU1_expAE05.nc',
 'VUW_PISM1_s1_expAE05.nc',
 'UNN_Ua_expAE05.nc',
 'DC_ISSM_expAE05.nc',
 'UTAS_ElmerIce_expAE05.nc',
 'NORCE_CISM2-MAR364-ERA-t1_expAE05.nc',
 'NORCE_CISM5-MAR364-ERA-t1-nonlocal_expAE05.nc',
 'LSCE_GRISLI_expAE05.nc',
 'IMAU_UFEMISM4_expAE05.nc',
 'NORCE_CISM3-MAR364-ERA-t1-local_expAE05.nc',
 'VUW_PISM1_s4_expAE05.nc',
 'VUW_PISM2_s1_expAE05.nc',
 'NORCE_CISM5-MAR364-ERA-t1-local_expAE05.nc',
 'NORCE_CISM4-MAR364-ERA-t1_expAE05.nc',
 'VUW_PISM2_s2_expAE05.nc',
 'ILTS_SICOPOLIS_expAE05.nc',
 'NORCE_CISM4-MAR364-JRA-t1_expAE05.nc',
 'UCSD_ISSM_expAE05.nc',
 'DOE_MALI_8km_AntMean_expAE05.nc',
 'NORCE

# Example create dataset for exp05 first model name and save on cluster

# 1D datasets:

### 1. average thermal forcing

In [3]:
# 1. rename the variable nanames from existing file like seroussi et al
variable_name = 'thermalforcing' #please don't change

filename =  f"{computed_name}/{expname}/shelfmelt/{file_exp[0]}"
d_example = xr.open_dataset(filename,decode_times=False )
d_calc =d_example.copy()

d_calc.attrs ={}

regions =['region_1','region_2','region_3']
sectors =[]
for i in range(18):
    sectors.append('sector_'+str(i+1))
#rename:
d_calc =d_calc.rename({'shelfmelt':variable_name })
for region in regions:
    varname_old = 'shelfmelt'+'_'+region
    varname_new = variable_name+'_'+region    
    d_calc =d_calc.rename({varname_old:varname_new })
for sector in sectors:
    varname_old = 'shelfmelt'+'_'+sector
    varname_new = variable_name+'_'+sector 
    d_calc =d_calc.rename({varname_old:varname_new })
   

In [4]:
    
#now add you values: 
# Loop through each variable in the dataset
for var_name in d_calc.variables:
    if (var_name != 'time') and (var_name != 'rhoi') and (var_name != 'rhow'):
        d_calc[var_name].values = np.zeros(len(d_example.time.values)) # assign you calc output not 0 :)


In [5]:
# save dataset 
path_save = f"{pth_calc_output}{expname}/computed_{variable_name}_{model_names_endfile[0]}"
d_calc.to_netcdf(path_save)

### 2. gl depth

In [16]:
variable_name = 'groundingline_depth'

### 3. gl TF

In [17]:
variable_name = 'groundingline_thermalforcing'

### 4. gl melt

In [ ]:
variable_name = 'groundingline_bmb'

### 5. dynamic SLR 

In [ ]:
variable_name = 'dynamic_slr'

# 2D datasets

## 6.  thermal forcing for depth-bin

In [6]:
# get depth bins which will be our new dimension for the dataset
d_z =  xr.open_dataset(pth_ismip6 +'/climatology_1995-2014/UKESM1-0-LL_SSP585_temperature_8km_x_60m.nc')
depth =d_z.z.values

In [18]:
# 1. rename the variable nanames 
variable_name = 'thermalforcing_dz' #please don't change

filename =  f"{computed_name}/{expname}/shelfmelt/{file_exp[0]}"
d_example = xr.open_dataset(filename,decode_times=False )
d_calc =d_example.copy()

d_calc.attrs ={}



regions =['region_1','region_2','region_3']
sectors =[]
for i in range(18):
    sectors.append('sector_'+str(i+1))
#rename:
d_calc =d_calc.rename({'shelfmelt':variable_name })
for region in regions:
    varname_old = 'shelfmelt'+'_'+region
    varname_new = variable_name+'_'+region    
    d_calc =d_calc.rename({varname_old:varname_new })
for sector in sectors:
    varname_old = 'shelfmelt'+'_'+sector
    varname_new = variable_name+'_'+sector 
    d_calc =d_calc.rename({varname_old:varname_new })
d_calc =d_calc.expand_dims(z=depth.size)




In [7]:
    
#now add you values: 
# Loop through each variable in the dataset
for var_name in d_calc.variables:
    
    if (var_name != 'time') and (var_name != 'rhoi') and (var_name != 'rhow'):
        zeros_array = np.zeros((depth.size, *d_calc[var_name].shape)) 
        d_calc[var_name] = (('z', *d_calc[var_name].dims), zeros_array)

d_calc =d_calc.assign_coords(z=depth)                                           

In [13]:
# save dataset 
path_save = f"{pth_calc_output}{expname}/computed_{variable_name}_{model_names_endfile[0]}"
d_calc.to_netcdf(path_save)

## 7. total basal melt for depth bin

In [19]:
variable_name = 'totalBMB_dz' #please don't change


## 8. average basal melt for depth bin

In [20]:
variable_name = 'averageBMB_dz'

## 9. aggregated shelf area for each depth bin

In [20]:
variable_name = 'totalshelfarea_dz'